# Welcome to QMCJu

Translated from QMCPy's qmcpy_intro.ipynb

This tutorial introduces QMCJu by walking through the four building blocks:
  1. Discrete Distribution — generates points in [0,1)^d
  2. True Measure — transforms points to the target domain
  3. Integrand — the function to integrate
  4. Stopping Criterion — decides when enough samples have been taken

In [ ]:
using QMCJu
using Statistics
using LinearAlgebra
using Printf

Importing QMCJu

In [ ]:
println("="^60)
println("QMCJu — Julia package for Quasi-Monte Carlo integration")
println("="^60)
println("  `using QMCJu` imports all types and functions.")
println()

IID vs Low-Discrepancy (LD) Sequences

In [ ]:
println("="^60)
println("IID vs Low-Discrepancy Sequences")
println("="^60)

dd_lat = Lattice(2; randomize=true, seed=7)
x_lat = gen_samples(dd_lat, 4)
println("  4 lattice points in 2D:")
for i in 1:4
    @printf("    [%.4f  %.4f]\n", x_lat[i,1], x_lat[i,2])
end
println()

dd_iid = IIDStdUniform(2; seed=7)
x_iid = gen_samples(dd_iid, 4)
println("  4 IID uniform points in 2D:")
for i in 1:4
    @printf("    [%.4f  %.4f]\n", x_iid[i,1], x_iid[i,2])
end
println("  LD points fill the space more evenly than IID.")
println()

Multi-Dimensional Inputs & Custom Functions

In [ ]:
println("="^60)
println("Custom Functions: ∫[0,1]^d ‖x‖^√(‖x‖) dx")
println("="^60)

## Define a custom integrand

Define a custom integrand that works with n×d matrices.
`x` is an n × d matrix; we compute ‖x‖ for each row, then return ‖x‖^√(‖x‖).

In [ ]:
function myfunc(x)
    # x is an n × d matrix; compute ‖x‖ for each row
    x_norms = sqrt.(sum(x .^ 2; dims=2))[:]
    # Avoid 0^0: replace zeros with a small value
    x_norms[x_norms .== 0.0] .= eps()
    return x_norms .^ sqrt.(x_norms)
end

d = 1

In [ ]:
println("  d = 1:")
dd = IIDStdUniform(1; seed=7)
tm = Uniform(dd)
f = CustomFun(tm, myfunc)
sc = CubMCCLT(f; abs_tol=0.05)
result = integrate(sc)
true_sol_1d = 0.658582  # Wolfram: ∫₀¹ x^√x dx
@printf("    QMCJu: %.6f, Exact: %.6f, Error: %.2e\n",
        result.solution, true_sol_1d, abs(result.solution - true_sol_1d))
println()

d = 2

In [ ]:
println("  d = 2:")
dd = IIDStdUniform(2; seed=7)
tm = Uniform(dd)
f = CustomFun(tm, myfunc)
sc = CubMCCLT(f; abs_tol=0.05)
result = integrate(sc)
true_sol_2d = 0.827606  # Wolfram: ∫₀¹∫₀¹ √(x²+y²)^√(√(x²+y²)) dxdy
@printf("    QMCJu: %.6f, Exact: %.6f, Error: %.2e\n",
        result.solution, true_sol_2d, abs(result.solution - true_sol_2d))
println()

The Four Building Blocks

In [ ]:
println("="^60)
println("The Four Building Blocks")
println("="^60)

println("""
  1. Discrete Distribution — generates low-discrepancy or IID points
     Available: IIDStdUniform, Lattice, DigitalNetB2, Halton

  2. True Measure — transforms [0,1)^d points to the integration domain
     Available: Uniform, Gaussian, BrownianMotion, Lebesgue,
                GeometricBrownianMotion, StudentT, Triangular,
                Kumaraswamy, JohnsonsSU, BernoulliCont

  3. Integrand — the function to integrate
     Available: CustomFun, Keister, Genz, AsianOption, FinancialOption,
                BoxIntegral, Linear0, Sin1D, Ishigami, Hartmann6D,
                Multimodal2D, FourBranch2D

  4. Stopping Criterion — decides when the estimate is accurate enough
     Available: CubMCCLT, CubQMCLatticeG, CubQMCNetG,
                CubQMCBayesLatticeG, CubQMCBayesNetG
""")

Putting It All Together

In [ ]:
println("="^60)
println("Putting It All Together: Genz Continuous Function")
println("="^60)

d = 5
a = ones(d)          # difficulty parameters
u = fill(0.5, d)     # shift parameters

IID MC

In [ ]:
dd = IIDStdUniform(d; seed=7)
tm = Uniform(dd)
f = Genz(tm; kind=:continuous, a=a, u=u)
exact = genz_exact(f)
sc = CubMCCLT(f; abs_tol=0.05)
result = integrate(sc)
@printf("  IID MC:        %.6f (exact = %.6f, n = %d)\n",
        result.solution, exact, result.data[:n])

Lattice QMC

In [ ]:
dd = Lattice(d; randomize=true, seed=7)
tm = Uniform(dd)
f = Genz(tm; kind=:continuous, a=a, u=u)
sc = CubQMCLatticeG(f; abs_tol=0.001, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Lattice QMC:   %.6f (exact = %.6f, n/rep = %d)\n",
        result.solution, exact, result.data[:n])

Sobol' QMC

In [ ]:
dd = DigitalNetB2(d; seed=7)
tm = Uniform(dd)
f = Genz(tm; kind=:continuous, a=a, u=u)
sc = CubQMCNetG(f; abs_tol=0.001, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Sobol' QMC:    %.6f (exact = %.6f, n/rep = %d)\n",
        result.solution, exact, result.data[:n])
println()

println("="^60)
println("QMCJu intro completed!")